# Alpaca: A Strong, Replicable Instruction-Following Model
`Alpaca 7B`，这是一个在 `52K` 指令跟随演示基础上从 `LLaMA 7B` 模型微调而来的模型。在我们对单轮指令跟随的初步评估中，`Alpaca` 表现得与 OpenAI 的 `text-davinci-003` 相似，同时出乎意料地小，而且容易/便宜复制（<600 美元）。

## 概述
像 GPT-3.5（text-davinci-003）、ChatGPT、Claude 和 Bing Chat 这样的遵循指令的模型变得越来越强大。然而，尽管它们被广泛使用，遵循指令的模型仍然存在许多缺陷：
- 可能生成虚假信息
- 传播社会刻板印象
- 产生有毒语言。

在学术界对遵循指令模型的研究一直很困难，因为没有任何易于获取的模型在能力上接近于封闭源模型，如 OpenAI 的 text-davinci-003。

因此，哈佛团队发布了按照指令微调方式得到的大语言模型：Alpaca，该模型是基于 Meta 的 LLaMA 7B 模型进行微调的。使用了52K指令微调数据训练 Alpaca 模型，这些指令是使用 text-davinci-003 以`Self-instruct`的方式生成的。在`Self-instruct`的评估数据集上，`Alpaca` 展示了许多与 OpenAI 的 `text-davinci-003` 相似的行为，并且模型非常容易复现。

注意：Alpaca是在2023年3月发布的模型，当时提出的动机在现在已经做得比Alpaca好了非常多。


## 训练方法
要训练一个高质量的指令跟随的模型，首先需要满足两个条件：
1. 强大的预训练语言模型。Alpaca选择了LLaMA模型。
2. 高质量的`instruction-following`数据。Alpaca提出了`Self-instruct`自动生成指令微调数据。

如下图所示，展示了`Alpaca`模型的训练方法。使用`Self-instruct`生成了`instruction-tuning`数据。

![Alt text](./_img/alpaca_train.png)



## Self-instruct
![Alt text](./_img/self_instruct.png)
`Self-instruct`是一个自动化生成指令微调数据的pipeline框架，用于提高预训练语言模型的指令跟随能力。它是一种迭代的自举算法，从一组手工编写的指令种子集开始，并使用其他能力较强的语言模型生成新的指令和相应的输入输出实例。这个框架使用语言模型生成指令、输入和输出样本，然后过滤无效的或类似的样本，然后使用它们来微调模型。

如上图所示，Self-instruct主要分为4个步骤：
1. 根据任务种子库生成新的指令任务
2. 对指令任务进行分类
3. 针对不同类型的指令生成输入输出
4. 质量过滤，过滤掉低质，重复，无效的数据

### Step 0 种子任务构造
进行自动化生成指令微调数据的前提是，有一个种子任务库。在开始构造之前，原文作者手动撰写了175个任务，每个任务有一个指令`instruction`，和一个样例`input&output`。

### Step 1 指令生成
`Self-instruct `是基于预训练大语言模型来构造新的指令，因为预训练模型的续写能力比较强，按照下面给出的模板输入多个`Task`任务指令，模型会根据给出的指令内容，续写一个新的指令。

每次生成从种子库中随机采样8个任务指令，其中6个是人类编写的，2个是在之前的步骤中生成的指令，按如下Prompt模板拼接：
![Alt text](./_img/instruction_generate.png)

### Step 2 分类任务识别
在全部任务中，需要特别注意的是分类任务，针对分类任务，需要先输出类别，再让模型根据指令和类别生成输入，否则模型容易生成倾向于某一个标签的样例。具体样例见Step3.

因此，第二步需要区分任务是否属于分类任务。原文作者使用GPT-3模型的`few-shot`能力来进行分类任务，构造的Prompt由12个分类指令和19个非分类指令组成，Prompt模板如下：
![Alt text](./_img/classify_prompt.png)



### Step 3 样例生成
接下来则是生成输入输出部分。

- 针对输入优先的非分类任务，Prompt模板如下：

![Alt text](./_img/instance_generate_input.png)


- 针对输出优先的分类任务，Prompt模板如下：
![Alt text](./_img/instance_generate.png)


### Step 4 数据过滤
为了提高指令微调数据的多样性，使用了以下几个标准对数据过滤：
1. 只有当生成的指令与其他所有指令的`ROUGE-L`相似度小于0.7时，才加入指令库中
2. 排除包含有特殊关键字的指令，例如images, pictures、graphs
3. 在为每个指令生成新实例时，过滤掉完全相同的实例或那些输入相同但输出不同的实例。
4. 基于启发式方法（例如，指令太长或太短，实例输出是输入的重复）识别并过滤掉无效的生成。

### 数据统计
原文作者使用这个方法生成了52K的指令instruction和82K的样例数据。

多样性统计如下图所示：
- 多样性：包括词性，句法树中最接近root的动词选取top10，每个动词最近的名词，选取top4，如图所示。与种子指令的ROUGE-L分布及指令长度分布如图所示。
![Alt text](./_img/data_gen_static.png)

## Instruction-tuning数据示例

In [ ]:
[
    {
        "instruction": "Give three tips for staying healthy.",
        "input": "",
        "output": "1. Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule."
    },
    {
        "instruction": "What are the three primary colors?",
        "input": "",
        "output": "The three primary colors are red, blue, and yellow."
    },
    {
        "instruction": "Make a prediction about what will happen in the next paragraph.",
        "input": "Mary had been living in the small town for many years and had never seen anything like what was coming.",
        "output": "It appears that something extraordinary and unprecedented will happen in the next paragraph."
    },
    {
        "instruction": "Generate a headline for the following article.",
        "input": "This article discusses the future of renewable energy sources in the US.",
        "output": "Renewable Energy Sources are the Future of the US"
    }
]

## 思考
### 1、为什么有效

为什么SELF-INSTRUCT能发挥作用？ 值得反思的是，高质量的人类反馈在促成最近关于指令调整LM的成功方面所发挥的作用究竟如何。

这里有两个极端的假说。

其一，人的反馈是指令调整的一个必要的和不可分割的方面，因为LM需要了解在预培训中没有完全学会的问题。

其二，人类反馈是指令调整的一个可选的方面，因为LM在预训练中已经对指令非常熟悉了。观察人类的反馈只是一个轻量级的过程，用于调整他们的预训练分布/目标，可能会被不同的过程所取代。 

虽然现实情况可能介于这两个极端之间，但实验猜测它更接近于𝐻2，特别是对于较大的模型。这种直觉，即LM已经知道很多语言指令，是SELF- INSTRUCT的一个关键动机，也得到了其经验性成功的支持。

此外，SELF-INSTRUCT可能有助于使广泛使用的指令调整模型（如InstructGPT）的 "幕后 "情况更加透明。

不幸的是，这些工业模型仍然在API墙后面，因为它们的数据集没有被公布，因此人们对它们的构造以及它们为什么表现出令人印象深刻的能力了解甚少。

### 2、SELF-INSTRUCT的局限性

首先，长尾现象。SELF-INSTRUCT依赖于LMs，它将继承LMs带来的所有限制。正如最近的研究表明（Razeghi等人，2022；Kandpal等人，2022），长尾现象对LMs的成功构成严重挑战。换句话说，LMs的最大收益对应于语言的频繁使用（语言使用分布的头部），而在低频语境中的收益最小。

同样，在这项工作的背景下，如果SELF-INSTRUCT的大部分收益都偏向于在预训练语料库中更频繁出现的任务或指令，那也不会令人惊讶。因此，该方法在不常见的和创造性的指令方面可能会表现得很脆弱。

其次，大型模型的依赖性。由于SELF- INSTRUCT依赖于从LM中提取的归纳偏见，它可能对较大的模型。如果是真的，这可能会对那些可能没有大型计算资源的人造成障碍。值得注意的是，有人类注释的指令调整也有类似的限制：指令调整的收益在较大的模型中更高（Wei等人，2022）。 

最后，强化LM的偏见。这种迭代算法会带来非预期后果。 迭代算法的意外后果，如放大有问题的社会偏见（对性别、种族等的刻板印象或诽谤）。与此相关的是，在这个过程中观察到的一个挑战是，该算法很难产生平衡的标签，这反映了模型的先前偏见。我们希望未来的工作能够解决这些细节问题，以更好地了解该方法的优点和缺点。

本段摘自：参考资料1

## 代码实践
官方代码：[https://github.com/yizhongw/self-instruct](https://github.com/yizhongw/self-instruct)

使用官方代码只需要按照顺序执行下面几个脚本即可：

In [ ]:
# 1. Generate instructions from the seed tasks
./scripts/generate_instructions.sh

# 2. Identify whether the instruction represents a classification task or not
./scripts/is_clf_or_not.sh

# 3. Generate instances for each instruction
./scripts/generate_instances.sh

# 4. Filtering, processing, and reformatting
./scripts/prepare_for_finetuning.sh

### Step 0
克隆代码仓库，仓库中有175个种子指令。

下面是调用GPT3的函数：

In [1]:
import json
import tqdm
import os
import random
import openai
from datetime import datetime
import time
import re
import string
import numpy as np
import pandas as pd
from multiprocessing import Pool
from functools import partial
from rouge_score import rouge_scorer
import glob
import pandas as pd
random.seed(42)

batch_dir ="data/gpt3_generations/"
seed_tasks_path ="data/seed_tasks.jsonl"
num_instructions_to_generate= 100
use_clf_seed_tasks_only = False
engine = "davinci"
num_prompt_instructions = 8
api_key = ""
template = "template_1"
request_batch_size = 5
max_instances_to_generate = 5
generation_tasks_only = False
classification_tasks_only = False
instance_files = ["data/batch_221203/machine_generated_instances.jsonl"]
classification_type_files = ["data/batch_221203/is_clf_or_not_davinci_template_1.jsonl"]
output_dir ="data/gpt3_generations/batch_221203/finetuning/"
num_instructions = 10
include_seed_tasks = False
seed_tasks_path = "data/seed_tasks.jsonl"


In [ ]:


def make_gpt3_requests(
        engine, prompts, max_tokens, temperature, top_p, 
        frequency_penalty, presence_penalty, stop_sequences, logprobs, n, best_of, retries=3, api_key=None, organization=None
    ):
    response = None
    target_length = max_tokens
    if api_key is not None:
        openai.api_key = api_key
    if organization is not None:
        openai.organization = organization
    retry_cnt = 0
    backoff_time = 30
    while retry_cnt <= retries:
        try:
            response = openai.Completion.create(
                engine=engine,
                prompt=prompts,
                max_tokens=target_length,
                temperature=temperature,
                top_p=top_p,
                frequency_penalty=frequency_penalty,
                presence_penalty=presence_penalty,
                stop=stop_sequences,
                logprobs=logprobs,
                n=n,
                best_of=best_of,
            )
            break
        except openai.error.OpenAIError as e:
            print(f"OpenAIError: {e}.")
            if "Please reduce your prompt" in str(e):
                target_length = int(target_length * 0.8)
                print(f"Reducing target length to {target_length}, retrying...")
            else:
                print(f"Retrying in {backoff_time} seconds...")
                time.sleep(backoff_time)
                backoff_time *= 1.5
            retry_cnt += 1
    
    if isinstance(prompts, list):
        results = []
        for j, prompt in enumerate(prompts):
            data = {
                "prompt": prompt,
                "response": {"choices": response["choices"][j * n: (j + 1) * n]} if response else None,
                "created_at": str(datetime.now()),
            }
            results.append(data)
        return results
    else:
        data = {
            "prompt": prompts,
            "response": response,
            "created_at": str(datetime.now()),
        }
        return [data]

### Step 1
生成新指令，代码如下：

- 构造Prompt

In [ ]:
def encode_prompt(prompt_instructions, classification=False):
    """Encode multiple prompt instructions into a single string."""
    if classification:
        prompt = "Come up with a series of classification tasks. Try to specify the possible output labels when possible.\n"
    else:
        prompt = "Come up with a series of tasks:\n"
    for idx, instruction in enumerate(prompt_instructions):
        instruction = re.sub(r"\s+", " ", instruction).strip().rstrip(":")
        prompt += f"{idx+1}. {instruction}\n"
    prompt += f"{len(prompt_instructions) + 1}."
    return prompt

- 随机采样指令

In [ ]:
def sample_machine_instructions(machine_instructions, similarities, n):
    """Sample n machine instructions from a list of machine instructions."""
    return random.sample(machine_instructions, min(n, len(machine_instructions)))

- 后处理

In [ ]:
def find_word_in_string(w, s):
    return re.compile(r'\b({0})\b'.format(w), flags=re.IGNORECASE).search(s)

def post_process_gpt3_response(response):
    if response is None or response["choices"][0]["finish_reason"] == "length":
        return []
    raw_instructions = re.split(r"\n\d+\s?\. ", response["choices"][0]["text"])
    instructions = []
    for inst in raw_instructions:
        inst = re.sub(r"\s+", " ", inst).strip()
        inst = inst.strip().capitalize()
        if inst == "":
            continue
        # filter out too short or too long instructions
        if len(inst.split()) <= 3 or len(inst.split()) > 150:
            continue
        # filter based on keywords that are not suitable for language models.
        if any(find_word_in_string(word, inst) for word in ["image", "images", "graph", "graphs", "picture", "pictures", "file", "files", "map", "maps", "draw", "plot", "go to"]):
            continue
        # We found that the model tends to add "write a program" to some existing instructions, which lead to a lot of such instructions.
        # And it's a bit comfusing whether the model need to write a program or directly output the result. 
        # Here we filter them out.
        # Note this is not a comprehensive filtering for all programming instructions.
        if inst.startswith("Write a program"):
            continue
        # filter those starting with punctuation
        if inst[0] in string.punctuation:
            continue
        # filter those starting with non-english character
        if not inst[0].isascii():
            continue
        instructions.append(inst)
    return instructions

- 生成新指令的主循环

In [ ]:
seed_tasks_path = "data/gpt3_generations/seed_tasks.jsonl" # seed_tasks文件路径
batch_dir = "data"
num_instructions_to_generate = 100
request_batch_size = 10
num_prompt_instructions = 8
use_clf_seed_tasks_only = False
engine = "gpt3"
api_key = ""

seed_tasks = [json.loads(l) for l in open(seed_tasks_path, "r")]

seed_instructions = [t["instruction"] for t in seed_tasks]
print(f"Loaded {len(seed_instructions)} human-written seed instructions")

os.makedirs(batch_dir, exist_ok=True)
request_idx = 0
# load the LM-generated instructions
machine_instructions = []
if os.path.exists(os.path.join(batch_dir, "machine_generated_instructions.jsonl")):
    with open(os.path.join(batch_dir, "machine_generated_instructions.jsonl"), "r") as fin:
        for line in fin:
            instruction_info = json.loads(line)
            machine_instructions.append(instruction_info["instruction"])
            request_idx = instruction_info["request_idx"] + 1
    print(f"Loaded {len(machine_instructions)} machine-generated instructions")

# similarities = {}
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

# now let's generate new instructions!
progress_bar = tqdm.tqdm(total=num_instructions_to_generate)
if machine_instructions:
    progress_bar.update(len(machine_instructions))

with open(os.path.join(batch_dir, "machine_generated_instructions.jsonl"), "a") as fout:
    while len(machine_instructions) < num_instructions_to_generate:
        batch_inputs = []
        for _ in range(request_batch_size):
            # sample machine instructions from the pool
            prompt_instructions = sample_machine_instructions(
                machine_instructions, 
                similarities=None,
                n=2)
            # sample human instructions from the pool
            prompt_instructions += random.sample(seed_instructions, num_prompt_instructions - len(prompt_instructions))
            random.shuffle(prompt_instructions)
            prompt = encode_prompt(prompt_instructions, classification=use_clf_seed_tasks_only)
            batch_inputs.append(prompt)
        results = make_gpt3_requests(
            engine=engine,
            prompts=batch_inputs,
            max_tokens=1024,
            temperature=0.7,
            top_p=0.5,
            frequency_penalty=0,
            presence_penalty=2,
            stop_sequences=["\n\n", "\n16", "16.", "16 ."],
            logprobs=1,
            n=1,
            best_of=1,
            api_key=api_key
        )
        instructions = []
        all_metadata = []
        for result in results:
            new_instructions = post_process_gpt3_response(result["response"])
            instructions += new_instructions
            all_metadata += [result] * len(new_instructions)

        for inst, metadata in zip(instructions, all_metadata):
            with Pool(4) as p:
                rouge_scores = p.map(partial(scorer.score, inst), seed_instructions + machine_instructions)
            rouge_scores = [score["rougeL"].fmeasure for score in rouge_scores]
            # rouge_scores = [scorer.score(inst, e_inst)["rougeL"].fmeasure for e_inst in human_instructions + machine_instructions]
            if max(rouge_scores) > 0.7:
                continue
            all_instructions = seed_instructions + machine_instructions
            most_similar_instructions = {
                    all_instructions[i] : rouge_scores[i] for i in np.argsort(rouge_scores)[-10:][::-1]
                }
            machine_instructions.append(inst)
            fout.write(json.dumps({
                "instruction": inst,
                "most_similar": most_similar_instructions,
                "avg_similarity_score": float(np.mean(rouge_scores)),
                "metadata": metadata,
                "request_idx": request_idx
            }) + "\n")
            progress_bar.update(1)
        request_idx += 1

### Step 2
判断是否为分类任务

- Prompt

In [ ]:
template_1 = '''Can the following task be regarded as a classification task with finite output labels?

Task: Given my personality and the job, tell me if I would be suitable.
Is it classification? Yes

Task: Give me an example of a time when you had to use your sense of humor.
Is it classification? No

Task: Replace the placeholders in the given text with appropriate named entities.
Is it classification? No

Task: Fact checking - tell me if the statement is true, false, or unknown, based on your knowledge and common sense.
Is it classification? Yes

Task: Return the SSN number for the person.
Is it classification? No

Task: Detect if the Reddit thread contains hate speech.
Is it classification? Yes

Task: Analyze the sentences below to identify biases.
Is it classification? No

Task: Select the longest sentence in terms of the number of words in the paragraph, output the sentence index.
Is it classification? Yes

Task: Find out the toxic word or phrase in the sentence.
Is it classification? No

Task: Rank these countries by their population.
Is it classification? No

Task: You are provided with a news article, and you need to identify all the categories that this article belongs to. Possible categories include: Music, Sports, Politics, Tech, Finance, Basketball, Soccer, Tennis, Entertainment, Digital Game, World News. Output its categories one by one, seperated by comma.
Is it classification? Yes

Task: Given the name of an exercise, explain how to do it.
Is it classification? No

Task: Select the oldest person from the list.
Is it classification? Yes

Task: Find the four smallest perfect numbers.
Is it classification? No

Task: Does the information in the document supports the claim? You can answer "Support" or "Unsupport".
Is it classification? Yes

Task: Create a detailed budget for the given hypothetical trip.
Is it classification? No

Task: Given a sentence, detect if there is any potential stereotype in it. If so, you should explain the stereotype. Else, output no.
Is it classification? No

Task: Explain the following idiom to me, and try to give me some examples.
Is it classification? No

Task: Is there anything I can eat for a breakfast that doesn't include eggs, yet includes protein, and has roughly 700-1000 calories?
Is it classification? No

Task: Answer the following multiple choice question. Select A, B, C, or D for the final answer.
Is it classification? Yes

Task: Decide whether the syllogism is logically sound.
Is it classification? Yes

Task: How can individuals and organizations reduce unconscious bias?
Is it classification? No

Task: What are some things you can do to de-stress?
Is it classification? No

Task: Find out the largest one from a set of numbers. Output the number directly.
Is it classification? Yes

Task: Replace the <mask> token in the text with proper words that are consistent with the context. You can use multiple words for each <mask> token.
Is it classification? No

Task: Write a cover letter based on the given facts.
Is it classification? No

Task: Identify the pos tag of the word in the given sentence.
Is it classification? Yes

Task: Write a program to compute the sum of integers from k to n.
Is it classification? No

Task: In this task, you need to compare the meaning of the two sentences and tell if they are the same. Output yes or no.
Is it classification? Yes

Task: To make the pairs have the same analogy, write the fourth word.
Is it classification? No

Task: Given a set of numbers, find all possible subsets that sum to a given number.
Is it classification? No

Task:'''

templates = {
    "template_1": template_1
}

- 主循环

In [ ]:

with open(os.path.join(batch_dir, "machine_generated_instructions.jsonl")) as fin:
    lines = fin.readlines()
    if num_instructions is not None:
        lines = lines[:num_instructions]

output_path = os.path.join(batch_dir, f"is_clf_or_not_{engine}_{template}.jsonl")
existing_requests = {}
if os.path.exists(output_path):
    with open(output_path) as fin:
        for line in tqdm.tqdm(fin):
            try:
                data = json.loads(line)
                existing_requests[data["instruction"]] = data
            except:
                pass
    print(f"Loaded {len(existing_requests)} existing requests")

progress_bar = tqdm.tqdm(total=len(lines))
with open(output_path, "w") as fout:
    for batch_idx in range(0, len(lines), request_batch_size):
        batch = [json.loads(line) for line in lines[batch_idx: batch_idx + request_batch_size]]
        if all(d["instruction"] in existing_requests for d in batch):
            for d in batch:
                data = existing_requests[d["instruction"]]
                data = OrderedDict(
                    (k, data[k]) for k in \
                        ["instruction", "is_classification"]
                    )
                fout.write(json.dumps(data, ensure_ascii=False) + "\n")
        else:
            # prefix = compose_prompt_prefix(human_written_tasks, batch[0]["instruction"], 8, 2)
            prefix = templates[template]
            prompts = [prefix + " " + d["instruction"].strip() + "\n" + "Is it classification?" for d in batch]
            results = make_gpt3_requests(
                engine=engine,
                prompts=prompts,
                max_tokens=3,
                temperature=0,
                top_p=0,
                frequency_penalty=0,
                presence_penalty=0,
                stop_sequences=["\n", "Task"],
                logprobs=1,
                n=1,
                best_of=1,
                api_key=api_key,
                organization=organization)
            for i in range(len(batch)):
                data = batch[i]
                if results[i]["response"] is not None:
                    data["is_classification"] = results[i]["response"]["choices"][0]["text"]
                else:
                    data["is_classification"] = ""
                data = {
                    "instruction": data["instruction"],
                    "is_classification": data["is_classification"]
                }
                data = OrderedDict(
                    (k, data[k]) for k in \
                        ["instruction", "is_classification"]
                    )
                fout.write(json.dumps(data, ensure_ascii=False) + "\n")
        progress_bar.update(len(batch))

### Step 3

In [ ]:
with open(os.path.join(batch_dir, input_file)) as fin:
    lines = fin.readlines()
    if num_instructions is not None:
        lines = lines[:num_instructions]
    tasks = []
    for line in lines:
        data = json.loads(line)
        if "metadata" in data:
            data["instruction_metadata"] = data["metadata"]
            del data["metadata"]
        tasks.append(data)

task_clf_types = {}
with open(os.path.join(batch_dir, "is_clf_or_not_davinci_template_1.jsonl")) as fin:
    for line in fin:
        data = json.loads(line)
        task_clf_types[data["instruction"]] = data["is_classification"].strip() in ["Yes", "yes", "YES"]

if classification_tasks_only:
    tasks = [task for task in tasks if task_clf_types[task["instruction"]]]

if generation_tasks_only:
    tasks = [task for task in tasks if not task_clf_types[task["instruction"]]]

output_path = os.path.join(batch_dir, output_file)
existing_requests = {}
if os.path.exists(output_path):
    with open(output_path) as fin:
        for line in tqdm.tqdm(fin):
            try:
                data = json.loads(line)
                existing_requests[data["instruction"]] = data
            except:
                pass
    print(f"Loaded {len(existing_requests)} existing requests")

progress_bar = tqdm.tqdm(total=len(tasks))
with open(output_path, "w") as fout:
    for batch_idx in range(0, len(tasks), request_batch_size):
        batch = tasks[batch_idx: batch_idx + request_batch_size]
        if all(d["instruction"] in existing_requests for d in batch):
            for d in batch:
                data = existing_requests[d["instruction"]]
                data = OrderedDict(
                    (k, data[k]) for k in \
                        ["instruction", "raw_instances", "instance_metadata", "instruction_metadata", 
                        "most_similar", "avg_similarity_score"]
                    )
                fout.write(json.dumps(data, ensure_ascii=False) + "\n")
        else:
            prompts = []
            for task in batch:
                if task_clf_types[task["instruction"]]:
                    prompt = output_first_template_for_clf + " " + task["instruction"].strip() + "\n"
                    prompts.append(prompt)
                else:
                    prompt = input_first_template_for_gen + " " + task["instruction"].strip() + "\n"
                    prompts.append(prompt)
            results = make_gpt3_requests(
                engine=engine,
                prompts=prompts,
                # because the clf template is longer, we need to decrease the max_tokens
                max_tokens=300 if any(task_clf_types[task["instruction"]] for task in batch) else 350,
                temperature=0,
                top_p=0,
                frequency_penalty=0,
                presence_penalty=1.5,
                stop_sequences=[f"Example {max_instances_to_generate + 1}", "Task:"],
                logprobs=1,
                n=1,
                best_of=1,
                api_key=api_key,
                organization=organization)
            for i in range(len(batch)):
                data = batch[i]
                data["instance_metadata"] = results[i]
                if results[i]["response"] is not None:
                    data["raw_instances"] = results[i]["response"]["choices"][0]["text"]
                else:
                    data["raw_instances"] = ""
                data = OrderedDict(
                    (k, data[k]) for k in \
                        ["instruction", "raw_instances", "instance_metadata", "instruction_metadata", 
                        "most_similar", "avg_similarity_score"]
                    )
                fout.write(json.dumps(data, ensure_ascii=False) + "\n")
        progress_bar.update(len(batch))

### Step 4
数据过滤

In [ ]:
def encode_instance(instruction, input, output, random_template=True):
    encoding_templates_w_input = [
        ("{instruction}\nInput: {input}\nOutput:", " {output}<|endoftext|>"),
        ("{instruction}\n\nInput: {input}\n\nOutput:", " {output}<|endoftext|>"),
        ("Task: {instruction}\nInput: {input}\nOutput:", " {output}<|endoftext|>"),
        ("{instruction}\n\n{input}\n\nOutput:", " {output}<|endoftext|>"),
        ("{instruction}\n\n{input}\n\n", "{output}<|endoftext|>"),
        ("{instruction}\n{input}\n\n", "{output}<|endoftext|>"),
        ("Task: {instruction}\n\n{input}\n\n", "{output}<|endoftext|>"),
    ]
    encoding_templates_wo_input = [
        ("{instruction} Output:", " {output}<|endoftext|>"),
        ("{instruction}\nOutput:", " {output}<|endoftext|>"),
        ("{instruction}\n\nOutput:", " {output}<|endoftext|>"),
        ("{instruction}\n", "{output}<|endoftext|>"),
        ("{instruction}\n\n", "{output}<|endoftext|>"),
        ("Task: {instruction}\n\n", "{output}<|endoftext|>"),
    ]
    if random_template:
        if input.strip() != "":
            prompt_template, completion_template = random.choice(encoding_templates_w_input)
            prompt = prompt_template.format(instruction=instruction.strip(), input=input.strip())
            completion = completion_template.format(output=output.strip())
        else:
            prompt_template, completion_template = random.choice(encoding_templates_wo_input)
            prompt = prompt_template.format(instruction=instruction.strip())
            completion = completion_template.format(output=output.strip())
    else:
        prompt = instruction.strip() + "\n\n" + input.strip() + "\n\n"
        completion = output.strip() + "<|endoftext|>"

    data = {
        "prompt": prompt,
        "completion": completion,
        "instruction": instruction.strip(),
        "input": input.strip(),
        "output": output.strip(),
    }
    return data


def parse_input_output(response_text):
    if re.findall(r"Output\s*\d*\s*:", response_text):
        inst_input = re.split(r"Output\s*\d*\s*:", response_text)[0].strip()
        inst_output = re.split(r"Output\s*\d*\s*:", response_text)[1].strip()
    else:
        inst_input = ""
        inst_output = response_text.strip()
    # to avoid the case multiple input/output pairs are generated
    if re.findall(r"Input\s*\d*\s*:", inst_output):
        inst_output = re.split(r"Input\s*\d*\s*:", inst_output)[0].strip()
    # remove the prefix "Input:" from the string
    inst_input = re.sub(r"^Input\s*\d*\s*:", "", inst_input).strip()
    return inst_input, inst_output


def filter_duplicate_instances(instances):
    # if the instances have same non-empty input, but different output, we will not use such instances
    same_input_diff_output = False
    for i in range(1, len(instances)):
        for j in range(0, i):
            if instances[i][1] == "":
                continue
            if instances[i][1] == instances[j][1] and instances[i][2] != instances[j][2]:
                same_input_diff_output = True
                break
    if same_input_diff_output:
        return []

    # remove duplicate instances
    instances = list(set(instances))
    return instances

def filter_invalid_instances(instances):
    filtered_instances = []
    for instance in instances:
        # if input and output are the same, we will not use such instances
        if instance[1] == instance[2]:
            continue
        # if output is empty, we will not use such instances
        if instance[2] == "":
            continue
        # if input or output ends with a colon, these are usually imcomplete generation. We will not use such instances
        if instance[1].strip().endswith(":") or instance[2].strip().endswith(":"):
            continue
        filtered_instances.append(instance)
    return filtered_instances

def parse_instances_for_generation_task(raw_text, instruction, response_metadata):
    instances = []
    raw_text = raw_text.strip()
    if re.findall("Example\s?\d*\.?", raw_text):
        instance_texts = re.split(r"Example\s?\d*\.?", raw_text)
        instance_texts = [it.strip() for it in instance_texts if it.strip() != ""]
        for instance_text in instance_texts:
            inst_input, inst_output = parse_input_output(instance_text)
            instances.append((instruction.strip(), inst_input.strip(), inst_output.strip()))
    elif re.findall(r"Output\s*\d*\s*:", raw_text):
        # we assume only one input/output pair in this case
        inst_input, inst_output = parse_input_output(raw_text)
        instances.append((instruction.strip(), inst_input.strip(), inst_output.strip()))
    else:
        return []
    # if the generation stops because of length, we remove the last instance
    if response_metadata["response"]["choices"][0]["finish_reason"] == "length":
        instances = instances[:-1]
    
    instances = filter_invalid_instances(instances)
    instances = filter_duplicate_instances(instances)
    return instances

def parse_instances_for_classification_task(raw_text, instruction, response_metadata):
    instances = []
    if not "Class label:" in raw_text:
        return []
    instance_texts = raw_text.split("Class label:")[1:]
    for instance_text in instance_texts:
        instance_text = instance_text.strip()
        fields = instance_text.split("\n", 1)
        if len(fields) == 2:
            # the first field split by \n is the class label
            class_label = fields[0].strip()
            # the rest is the input
            input_text = fields[1].strip()
        elif len(fields) == 1:
            # the first field split by \n is the input
            class_label = fields[0].strip()
            input_text = ""
        else:
            raise ValueError("Invalid instance text: {}".format(instance_text))
        instances.append((instruction.strip(), input_text.strip(), class_label.strip()))

    # if the generation stops because of length, we remove the last instance
    if response_metadata["response"]["choices"][0]["finish_reason"] == "length":
        instances = instances[:-1]
    instances = filter_invalid_instances(instances)
    instances = filter_duplicate_instances(instances)
    return instances


if __name__ == "__main__":
    args = parse_args()

    training_instances = []
    
    generated_tasks = []
    for instance_file in args.instance_files:
        with open(instance_file) as fin:
            for line in fin:
                generated_tasks.append(json.loads(line))
    print(f"Loaded {len(generated_tasks)} raw generated tasks")

    task_clf_types = {}
    for file in args.classification_type_files:
        with open(file) as fin:
            for line in fin:
                data = json.loads(line)
                task_clf_types[data["instruction"]] = data["is_classification"].strip() in ["Yes", "yes", "YES"]

    for task in tqdm.tqdm(generated_tasks):
        # get instruction
        instruction = task["instruction"]
        task["is_classification"] = task_clf_types[instruction]

        # get the instances
        if task["is_classification"]:
            task_instances = parse_instances_for_classification_task(task["raw_instances"], instruction, task["instance_metadata"])
        else:
            task_instances = parse_instances_for_generation_task(task["raw_instances"], instruction, task["instance_metadata"])

        # we only allow max 5 instances per task
        task_instances = random.sample(task_instances, min(len(task_instances), 5))
        
        if not task_instances:
            continue

        training_instances += task_instances


    os.makedirs(args.output_dir, exist_ok=True)
    with open(os.path.join(args.output_dir, "all_generated_instances.jsonl"), "w") as fout:
        for instance in training_instances:
            fout.write(json.dumps({
                "instruction": instance[0],
                "input": instance[1],
                "output": instance[2],
            }) + "\n")
    print(f"Saved {len(training_instances)} instances")
    unique_instructions = set([it[0] for it in training_instances])
    print(f"Unique instructions: {len(unique_instructions)}")
    clf_instructions = [instruction for instruction in unique_instructions if task_clf_types[instruction]]
    print(f"Classification instructions: {len(clf_instructions)}")
    non_clf_instructions = [instruction for instruction in unique_instructions if not task_clf_types[instruction]]
    print(f"Non-classification instructions: {len(non_clf_instructions)}")

    if args.num_instructions is not None:
        print(f"Sampling {args.num_instructions} instructions")
        sampled_instructions = random.sample(unique_instructions, args.num_instructions)
        training_instances = [it for it in training_instances if it[0] in sampled_instructions]
        print(f"Only using {len(training_instances)} instances for these sampled instructions.")
        with open(os.path.join(args.output_dir, f"sampled_generated_instances_{args.num_instructions}.jsonl"), "w") as fout:
            for instance in training_instances:
                fout.write(json.dumps({
                    "instruction": instance[0],
                    "input": instance[1],
                    "output": instance[2],
                }) + "\n")

    if args.include_seed_tasks:
        seed_tasks = [json.loads(l) for l in open(args.seed_tasks_path, "r")]
        for task in seed_tasks:
            for instance in task["instances"]:
                training_instances.append((task["instruction"], instance["input"], instance["output"]))
        print(f"Included {len(seed_tasks)} seed tasks")

    # get the prompt and completion for training gpt3
    gpt3_instances = []
    for instance in training_instances:
        # get input and do preprocessing
        inst_input = instance[1]
        # for some tasks, we check whether the input contains colon, and if so, we remove the part before the colon
        if random.random() < 0.5:
            colon_words = re.findall(r"(\w+):", inst_input)
            # if only one colon is found, we assume the instance only have one input and we remove the field name before the colon
            if len(set(colon_words)) == 1:
                inst_input = inst_input.split(":", 1)[1].strip()
            else:
                inst_input = inst_input.strip()
            # we also replace two consecutive new lines with one new line half of the time
            inst_input = inst_input.replace("\n\n", "\n")
        
        gpt3_instances.append(encode_instance(instance[0], inst_input, instance[2]))

    # remove duplicates
    filtered_instances = []
    prompt_completion_set = set()
    for instance in gpt3_instances:
        instance_pair = (instance["prompt"], instance["completion"])
        if instance_pair not in prompt_completion_set:
            prompt_completion_set.add((instance["prompt"], instance["completion"]))
            filtered_instances.append(instance)
    gpt3_instances = filtered_instances

    # shuffle
    random.shuffle(gpt3_instances)
    with open(os.path.join(args.output_dir, f"gpt3_finetuning_data_{len(gpt3_instances)}.jsonl"), "w") as fout:
        for instance in gpt3_instances:
            fout.write(json.dumps({
                "prompt": instance["prompt"],
                "completion": instance["completion"],
            }) + "\n")

# Alpaca训练教程


## 参考资料
1. [面向大模型微调的instruction指令自动化生成技术：SELF-INSTRUCT指令自动化生成框架工作介绍](https://mp.weixin.qq.com/s/QW-dGXUsrHXFdIfGI68HXw)
2. [SELF-INSTRUCT: Aligning Language Models with Self-Generated Instructions](https://arxiv.org/pdf/2212.10560)